# MagXMCD — Inverse Scattering: CNN vs Classical Phase Retrieval
### Corrected XMCD-difference physics · constant electron density · 7-way benchmark

**Measurement (no holography):**
$$I^- = I_R - I_L = 4\,\mathrm{Re}[A_c^* A_m],\qquad A_m=\chi\,\mathrm{FFT}[\hat k\cdot M]$$
Electron density treated as **constant** (flat reference $A_c=C$) for the first runs;
switchable later. The one lever for a better fit is **more $(\theta,\phi)$ angles**.

**Seven pipelines on the same data:** CNN alone · RAAR/GENFIRE/RESIRE alone ·
RAAR/GENFIRE/RESIRE → CNN (warm-start helper).

**Classical methods are GPU-batched**

**Figures (every pipeline):** per-method test run (true vector field → I⁻ scattering →
reconstructed field), accuracy table, master comparison grid, accuracy bar chart,
training curves, and exported CSV/NPZ.

In [1]:
import tensorflow as tf
print("TF",tf.__version__); g=tf.config.list_physical_devices('GPU')
print("GPU:",g[0].name if g else "NONE — Runtime>GPU")

TF 2.20.0
GPU: /physical_device:GPU:0


## Config

In [2]:
import os,time,json,csv,warnings
os.environ['TF_CPP_MIN_LOG_LEVEL']='2'; warnings.filterwarnings('ignore')
import numpy as np, matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
import tensorflow as tf
from tensorflow import keras; from tensorflow.keras import layers

HC_EV_NM=1239.84; PHOTON_EV=707.0
LAMBDA_NM=HC_EV_NM/PHOTON_EV; K_MAG=2*np.pi/LAMBDA_NM; XMCD_ASYM=0.3
GRID_N=64; SAMPLE_NM=500.0; DX_NM=SAMPLE_NM/GRID_N
C_CONST=1.0                       # constant electron density reference
LAMBDA_DOM_NM=(250,450); XI_DOM_NM=(80,200); WALL_TYPE='both'

# angle lever
THETA_LIST=np.linspace(15,75,4)
PHI_LIST=np.linspace(0,360,6,endpoint=False)
ANGLE_LIST=[(float(t),float(p)) for t in THETA_LIST for p in PHI_LIST]
N_ANGLES=len(ANGLE_LIST)

N_TRAIN=3000; N_VAL=750; N_TEST=300
N_EVAL_CLASSICAL=150              # classical methods run on this many (GPU-batched, fast)
BATCH_SIZE=32; EPOCHS=30; LR=3e-4
RAAR_ITERS=150; GENFIRE_ITERS=120; RESIRE_ITERS=120; RESIRE_LR=0.5; RAAR_BETA=0.9
CLF_BATCH=64                     # GPU batch for classical methods

print(f"angles {N_ANGLES} | train {N_TRAIN} | classical eval {N_EVAL_CLASSICAL}")


angles 24 | train 3000 | classical eval 150


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Section 1 — Domain generator (Bloch + Néel walls)

In [4]:
def generate_domain(N,rng,wall_type=WALL_TYPE,lambda_nm=None,xi_nm=None):
    if lambda_nm is None: lambda_nm=rng.uniform(*LAMBDA_DOM_NM)
    if xi_nm is None: xi_nm=rng.uniform(*XI_DOM_NM)
    qx=np.fft.fftfreq(N,d=DX_NM)*2*np.pi; QX,QY=np.meshgrid(qx,qx)
    Q=np.sqrt(QX**2+QY**2); Q[0,0]=1e-12
    f=np.real(np.fft.ifft2(np.fft.fft2(rng.standard_normal((N,N)))*np.exp(-0.5*((Q-2*np.pi/lambda_nm)/(1/xi_nm))**2)))
    Mz=np.sign(f).astype(np.float32); Mz[Mz==0]=1
    gy,gx=np.gradient(f); gm=np.sqrt(gx**2+gy**2)+1e-12; nx,ny=gx/gm,gy/gm
    fn=f/(np.abs(f).max()+1e-12); ww=np.exp(-(fn/0.15)**2)
    tilt=rng.uniform(0.3,0.7); c=rng.choice([-1.,1.]); wt=wall_type
    if wt=='both': wt=rng.choice(['neel','bloch'])
    if wt=='neel': Mx=c*tilt*ww*nx; My=c*tilt*ww*ny
    else: Mx=c*tilt*ww*(-ny); My=c*tilt*ww*nx
    Mze=Mz*np.sqrt(np.clip(1-(tilt*ww)**2,0,1))
    mg=np.sqrt(Mx**2+My**2+Mze**2); mg[mg<1e-12]=1
    return (Mx/mg).astype(np.float32),(My/mg).astype(np.float32),(Mze/mg).astype(np.float32),dict(wall=wt)
print("generator ready")


generator ready


## Section 2 — Forward model: XMCD difference, constant density

$I^-=4C\,\mathrm{Re}[A_m]$, $A_m=\chi\,\mathrm{FFT}[\hat k\cdot M]$ — linear in M.

In [5]:
def k_hat(th,ph):
    t,p=np.deg2rad(th),np.deg2rad(ph)
    return np.array([np.sin(t)*np.cos(p),np.sin(t)*np.sin(p),-np.cos(t)])
def A_mag(Mx,My,Mz,th,ph):
    kh=k_hat(th,ph); proj=kh[0]*Mx+kh[1]*My+kh[2]*Mz
    return XMCD_ASYM*np.fft.fftshift(np.fft.fft2(proj))*DX_NM**2
def Iminus(Mx,My,Mz,th,ph):
    return (4*C_CONST*np.real(A_mag(Mx,My,Mz,th,ph))).astype(np.float32)
def build_input(Mx,My,Mz,log_scale=True):
    ch=[]
    for (th,ph) in ANGLE_LIST:
        b=Iminus(Mx,My,Mz,th,ph)
        if log_scale: b=np.sign(b)*np.log1p(np.abs(b))
        p99=np.percentile(np.abs(b),99)
        if p99>1e-12: b=np.clip(b/p99,-1,1)
        ch.append(b.astype(np.float32))
    return np.stack(ch,axis=-1)
print(f"forward model ready, {N_ANGLES} I⁻ channels")


forward model ready, 24 I⁻ channels


## Section 3 — Diagnostic figure

In [6]:
rng=np.random.default_rng(1); Mx,My,Mz,p=generate_domain(GRID_N,rng,lambda_nm=350,xi_nm=120)
fig=plt.figure(figsize=(20,4)); fig.patch.set_facecolor('#0d1117')
gs=gridspec.GridSpec(1,5,wspace=0.35)
def dk(ax):
    ax.set_facecolor('#161b22'); ax.tick_params(colors='#c9d1d9',labelsize=8)
    for s in ax.spines.values(): s.set_color('#30363d')
    return ax
ax=dk(fig.add_subplot(gs[0])); im=ax.imshow(Mz,cmap='RdBu_r',vmin=-1,vmax=1,origin='lower'); plt.colorbar(im,ax=ax,fraction=0.046)
s=4; xs=np.arange(0,GRID_N,s); XX,YY=np.meshgrid(xs,xs)
ax.quiver(XX,YY,Mx[::s,::s],My[::s,::s],color='k',alpha=0.5,scale=8)
ax.set_title(f'Mz + walls ({p["wall"]})',color='#e6edf3',fontsize=9); ax.axis('off')
for ci,(th,ph) in enumerate([(15,0),(35,90),(55,0),(75,180)]):
    Im=Iminus(Mx,My,Mz,th,ph); ax=dk(fig.add_subplot(gs[ci+1]))
    v=np.percentile(np.abs(Im),99); im=ax.imshow(Im,cmap='RdBu_r',vmin=-v,vmax=v,origin='lower'); plt.colorbar(im,ax=ax,fraction=0.046)
    ax.set_title(f'I⁻ θ={th:.0f} φ={ph:.0f}',color='#e6edf3',fontsize=9); ax.axis('off')
fig.suptitle('Diagnostic — XMCD difference I⁻(q)',color='#e6edf3',fontsize=12,fontweight='bold')
plt.savefig('fig0_diagnostic.png',dpi=150,bbox_inches='tight',facecolor=fig.get_facecolor()); plt.show()
print("saved fig0_diagnostic.png")


saved fig0_diagnostic.png


## Section 4 — Dataset

In [7]:
def gen_dataset(n,seed):
    rng=np.random.default_rng(seed)
    X=np.zeros((n,GRID_N,GRID_N,N_ANGLES),np.float32); Y=np.zeros((n,GRID_N,GRID_N,3),np.float32)
    t0=time.time()
    for i in range(n):
        mx,my,mz,_=generate_domain(GRID_N,rng); X[i]=build_input(mx,my,mz)
        Y[i,:,:,0],Y[i,:,:,1],Y[i,:,:,2]=mx,my,mz
        if (i+1)%500==0:
            r=(i+1)/(time.time()-t0); print(f"  {i+1}/{n} ETA {(n-i-1)/r:.0f}s")
    return X,Y
print("generating data...")
t0=time.time()
X_train,Y_train=gen_dataset(N_TRAIN,42); X_val,Y_val=gen_dataset(N_VAL,99); X_test,Y_test=gen_dataset(N_TEST,137)
print(f"done {time.time()-t0:.0f}s  X{X_train.shape} Y{Y_train.shape}")


generating data...
  500/3000 ETA 50s
  1000/3000 ETA 34s
  1500/3000 ETA 25s
  2000/3000 ETA 16s
  2500/3000 ETA 8s
  3000/3000 ETA 0s
  500/750 ETA 4s
done 63s  X(3000, 64, 64, 24) Y(3000, 64, 64, 3)


## Section 5 — CNN (Attention U-Net)

In [8]:
def rb(x,f):
    sk=x
    if x.shape[-1]!=f: sk=layers.Conv2D(f,1,padding='same')(x)
    x=layers.Conv2D(f,3,padding='same',use_bias=False)(x); x=layers.BatchNormalization()(x); x=layers.Activation('gelu')(x)
    x=layers.Conv2D(f,3,padding='same',use_bias=False)(x); x=layers.BatchNormalization()(x); x=layers.Add()([x,sk])
    return layers.Activation('gelu')(x)
def ag(x,g,f):
    tx=layers.Conv2D(f,1,padding='same')(x); pg=layers.Conv2D(f,1,padding='same')(g)
    psi=layers.Conv2D(1,1,activation='sigmoid',padding='same')(layers.Activation('relu')(layers.Add()([tx,pg])))
    return layers.Multiply()([x,psi])
def build_model(n_ch,N=GRID_N):
    inp=keras.Input((N,N,n_ch))
    e1=rb(inp,32);e1=rb(e1,32);p1=layers.MaxPooling2D()(e1)
    e2=rb(p1,64);e2=rb(e2,64);p2=layers.MaxPooling2D()(e2)
    e3=rb(p2,128);e3=rb(e3,128);p3=layers.MaxPooling2D()(e3)
    b=rb(p3,256);b=rb(b,256)
    u3=layers.UpSampling2D()(b);u3=layers.Concatenate()([u3,ag(e3,u3,64)]);d3=rb(u3,128);d3=rb(d3,128)
    u2=layers.UpSampling2D()(d3);u2=layers.Concatenate()([u2,ag(e2,u2,32)]);d2=rb(u2,64);d2=rb(d2,64)
    u1=layers.UpSampling2D()(d2);u1=layers.Concatenate()([u1,ag(e1,u1,16)]);d1=rb(u1,32);d1=rb(d1,32)
    return keras.Model(inp,layers.Conv2D(3,1,activation='tanh')(d1))
def train_cnn(Xtr,Ytr,Xva,Yva,n_ch,tag,epochs=EPOCHS):
    m=build_model(n_ch)
    sched=keras.optimizers.schedules.CosineDecay(LR,epochs*(len(Xtr)//BATCH_SIZE),alpha=0.05)
    opt=keras.optimizers.Adam(sched)
    @tf.function
    def ts(xb,yb):
        with tf.GradientTape() as t: yp=m(xb,training=True); l=tf.reduce_mean((yb-yp)**2)
        g=t.gradient(l,m.trainable_variables); opt.apply_gradients(zip(g,m.trainable_variables)); return l
    @tf.function
    def vs(xb,yb): return tf.reduce_mean((yb-m(xb,training=False))**2)
    best=np.inf; bw=None; H={'tr':[],'va':[]}; t0=time.time()
    print(f"[{tag}] {epochs}ep {len(Xtr)}smp {n_ch}ch")
    for ep in range(1,epochs+1):
        idx=np.random.permutation(len(Xtr))
        tl=[float(ts(tf.constant(Xtr[idx[b:b+BATCH_SIZE]]),tf.constant(Ytr[idx[b:b+BATCH_SIZE]]))) for b in range(0,len(Xtr),BATCH_SIZE)]
        vl=[float(vs(tf.constant(Xva[b:b+BATCH_SIZE]),tf.constant(Yva[b:b+BATCH_SIZE]))) for b in range(0,len(Xva),BATCH_SIZE)]
        H['tr'].append(np.mean(tl)); H['va'].append(np.mean(vl))
        if np.mean(vl)<best: best=np.mean(vl); bw=[w.numpy() for w in m.weights]
        if ep%5==0 or ep==1: print(f"  ep{ep} tr{np.mean(tl):.4f} va{np.mean(vl):.4f} {(time.time()-t0)/60:.1f}m")
    if bw:
        for w,v in zip(m.weights,bw): w.assign(v)
    print(f"  [{tag}] best va {best:.4f}")
    return m,H
print("CNN ready")


CNN ready


## Section 6 — Classical methods (GPU-batched): RAAR, GENFIRE, RESIRE

All operate on the I⁻ stack to recover Mz. Batched on GPU via TensorFlow FFTs —
reconstruct `CLF_BATCH` samples at once, hundreds of times faster than serial numpy.

In [9]:
# Precompute angle geometry as tensors
_K=tf.constant(np.array([k_hat(t,p) for (t,p) in ANGLE_LIST]),tf.float32)  # [na,3]
_Kpinv=tf.constant(np.linalg.pinv(np.array([k_hat(t,p) for (t,p) in ANGLE_LIST])),tf.float32)  # [3,na]

def _stack_to_ReFM(Im_stack_b):
    # Im_stack_b: [B,na,N,N] -> ReFM [B,N,N,3] via least squares K@ReFM=b per q
    B=tf.shape(Im_stack_b)[0]; na=len(ANGLE_LIST)
    b=Im_stack_b/(4*C_CONST*XMCD_ASYM*DX_NM**2)         # = k·ReFM per angle
    b=tf.transpose(b,[0,2,3,1])                          # [B,N,N,na]
    ReFM=tf.einsum('cn,bxyn->bxyc',_Kpinv,b)             # [B,N,N,3]
    return ReFM

def classical_recon_batch(Im_stack_b, method='raar', iters=None):
    """Im_stack_b: tf [B,na,N,N]. Returns Mz estimate [B,N,N] (GPU)."""
    ReFM=_stack_to_ReFM(Im_stack_b)        # [B,N,N,3]; channel 2 = Re[FFT[Mz_proj]]
    ReFMz=ReFM[...,2]                       # [B,N,N]
    N=GRID_N
    if method=='direct':
        obj=tf.math.real(tf.signal.ifft2d(tf.cast(tf.signal.ifftshift(ReFMz,axes=[1,2]),tf.complex64)))/DX_NM**2
        return obj
    # iterative: enforce known Re[FFT] each iter, real-space amplitude constraint
    if iters is None:
        iters={'raar':RAAR_ITERS,'genfire':GENFIRE_ITERS,'resire':RESIRE_ITERS}[method]
    ReF=tf.cast(ReFMz,tf.complex64)
    obj=tf.zeros_like(ReFMz)
    for it in range(iters):
        F=tf.signal.fftshift(tf.signal.fft2d(tf.cast(obj,tf.complex64)),axes=[1,2])
        if method=='genfire':
            F=tf.complex(tf.math.real(ReF),tf.math.imag(F))     # enforce known real part
            obj=tf.math.real(tf.signal.ifft2d(tf.signal.ifftshift(F,axes=[1,2])))
            obj=tf.clip_by_value(obj,-1,1)
        elif method=='resire':
            resid=tf.math.real(F)-tf.math.real(ReF)
            grad=tf.math.real(tf.signal.ifft2d(tf.signal.ifftshift(tf.cast(resid,tf.complex64),axes=[1,2])))
            obj=tf.clip_by_value(obj-RESIRE_LR*grad,-1,1)
        else:  # raar (proper reflector update on real-part constraint + support)
            Fm=tf.complex(tf.math.real(ReF),tf.math.imag(F))
            gm=tf.math.real(tf.signal.ifft2d(tf.signal.ifftshift(Fm,axes=[1,2])))  # P_M
            ref_m=2*gm-obj                                       # R_M
            ps=tf.clip_by_value(ref_m,-1,1)                      # P_S R_M (support/amp)
            obj=0.5*RAAR_BETA*(2*ps - ref_m + obj)+(1-RAAR_BETA)*gm  # RAAR
    return obj

def get_Im_stack_np(Mx,My,Mz):
    return np.stack([Iminus(Mx,My,Mz,t,p) for (t,p) in ANGLE_LIST],0)

def run_classical_on_set(Y, method, n):
    """Run a classical method on first n samples of Y. GPU-batched. Returns Mz preds [n,N,N]."""
    preds=[]; t0=time.time()
    for b in range(0,n,CLF_BATCH):
        idx=range(b,min(b+CLF_BATCH,n))
        Imb=np.stack([get_Im_stack_np(Y[i,:,:,0],Y[i,:,:,1],Y[i,:,:,2]) for i in idx],0)  # [B,na,N,N]
        out=classical_recon_batch(tf.constant(Imb,tf.float32),method).numpy()
        preds.append(out)
    preds=np.concatenate(preds,0)
    print(f"  [{method}] {n} samples in {time.time()-t0:.1f}s (GPU-batched)")
    return preds
print("classical methods ready (GPU-batched)")


classical methods ready (GPU-batched)


## Section 7 — Train CNN alone

In [10]:
model_cnn,hist_cnn=train_cnn(X_train,Y_train,X_val,Y_val,N_ANGLES,'cnn_alone')

[cnn_alone] 30ep 3000smp 24ch
  ep1 tr0.3769 va0.3339 0.9m
  ep5 tr0.3354 va0.3336 2.0m
  ep10 tr0.3326 va0.3351 3.6m
  ep15 tr0.2020 va0.4147 5.1m
  ep20 tr0.1005 va0.5145 6.7m
  ep25 tr0.0671 va0.5572 8.3m
  ep30 tr0.0558 va0.5781 9.9m
  [cnn_alone] best va 0.3335


## Section 8 — Helper datasets (classical → CNN), GPU-batched

Each classical method's Mz estimate is appended as an extra input channel.


In [11]:
def build_helper_channel(Y, method, n):
    """Compute classical Mz estimate for n samples, GPU-batched -> [n,N,N] normalized."""
    est=run_classical_on_set(Y,method,n)
    m=np.abs(est).max(axis=(1,2),keepdims=True); m[m<1e-12]=1
    return (est/m).astype(np.float32)

def augment_with_helper(X, helper):
    Xh=np.zeros((X.shape[0],GRID_N,GRID_N,N_ANGLES+1),np.float32)
    Xh[...,:N_ANGLES]=X; Xh[...,N_ANGLES]=helper
    return Xh

helpers=['raar','genfire','resire']
helper_models={}; helper_hist={}
for name in helpers:
    print(f"\n=== helper: {name} ===")
    htr=build_helper_channel(Y_train,name,N_TRAIN)
    hva=build_helper_channel(Y_val,name,N_VAL)
    Xtr_h=augment_with_helper(X_train,htr); Xva_h=augment_with_helper(X_val,hva)
    m,H=train_cnn(Xtr_h,Y_train,Xva_h,Y_val,N_ANGLES+1,f'{name}+cnn')
    helper_models[name]=m; helper_hist[name]=H



=== helper: raar ===
  [raar] 3000 samples in 73.4s (GPU-batched)
  [raar] 750 samples in 16.8s (GPU-batched)
[raar+cnn] 30ep 3000smp 25ch


KeyboardInterrupt: 

## Section 9 — Evaluate all 7 pipelines on the same test set

In [ ]:
eps=1e-7
def dacc(pmz,tmz): return (np.sign(pmz)==np.sign(tmz)).mean()
def vang(Yp,Yt):
    nt=np.linalg.norm(Yt,axis=-1,keepdims=True)+eps; npd=np.linalg.norm(Yp,axis=-1,keepdims=True)+eps
    c=np.clip(np.sum((Yt/nt)*(Yp/npd),-1),-1+eps,1-eps); return np.degrees(np.arccos(c)).mean()
results={}
# 1 CNN alone
P=model_cnn.predict(X_test,verbose=0)
results['CNN alone']={'mz':P[...,2],'full':P,
  'acc':np.mean([dacc(P[i,:,:,2],Y_test[i,:,:,2]) for i in range(N_TEST)]),
  'ang':np.mean([vang(P[i],Y_test[i]) for i in range(N_TEST)]),'mse':np.mean((P-Y_test)**2)}
# 2-4 classical alone (on eval subset)
ne=N_EVAL_CLASSICAL
for name in helpers:
    est=run_classical_on_set(Y_test,name,ne)
    accs=[dacc(est[i],Y_test[i,:,:,2]) for i in range(ne)]
    full=np.zeros((ne,GRID_N,GRID_N))  # mz only
    results[f'{name.upper()} alone']={'mz':est,'acc':np.mean(accs),'ang':None,'mse':None,'neval':ne}
# 5-7 classical->CNN
for name in helpers:
    hte=build_helper_channel(Y_test,name,N_TEST); Xte_h=augment_with_helper(X_test,hte)
    P=helper_models[name].predict(Xte_h,verbose=0)
    results[f'{name.upper()}→CNN']={'mz':P[...,2],'full':P,
      'acc':np.mean([dacc(P[i,:,:,2],Y_test[i,:,:,2]) for i in range(N_TEST)]),
      'ang':np.mean([vang(P[i],Y_test[i]) for i in range(N_TEST)]),'mse':np.mean((P-Y_test)**2)}
print("="*62)
print(f"  {'METHOD':<16}{'Mz acc':<12}{'vec ang':<12}{'MSE'}")
print("="*62)
for k,v in results.items():
    a=f"{v['acc']*100:.1f}%"; ang=f"{v['ang']:.1f}°" if v['ang'] else "—"; ms=f"{v['mse']:.4f}" if v['mse'] else "—"
    print(f"  {k:<16}{a:<12}{ang:<12}{ms}")
print("="*62)


## Section 10 — Per-method test-run figures
True vector field → I⁻ scattering → reconstructed field, for each of the 7.

In [ ]:
ti=0; mx,my,mz=Y_test[ti,:,:,0],Y_test[ti,:,:,1],Y_test[ti,:,:,2]
Im0=Iminus(mx,my,mz,35,0); s=4; xs=np.arange(0,GRID_N,s); XX,YY=np.meshgrid(xs,xs)
for method,res in results.items():
    fig=plt.figure(figsize=(15,4.5)); fig.patch.set_facecolor('#0d1117'); gs=gridspec.GridSpec(1,3,wspace=0.3)
    ax=fig.add_subplot(gs[0]); ax.set_facecolor('#161b22')
    im=ax.imshow(mz,cmap='RdBu_r',vmin=-1,vmax=1,origin='lower'); plt.colorbar(im,ax=ax,fraction=0.046)
    ax.quiver(XX,YY,mx[::s,::s],my[::s,::s],color='k',alpha=0.5,scale=8)
    ax.set_title('TRUE field',color='#e6edf3',fontsize=10); ax.axis('off')
    ax=fig.add_subplot(gs[1]); ax.set_facecolor('#161b22')
    v=np.percentile(np.abs(Im0),99); im=ax.imshow(Im0,cmap='RdBu_r',vmin=-v,vmax=v,origin='lower'); plt.colorbar(im,ax=ax,fraction=0.046)
    ax.set_title('SCATTERING I⁻',color='#e6edf3',fontsize=10); ax.axis('off')
    ax=fig.add_subplot(gs[2]); ax.set_facecolor('#161b22')
    mzp=res['mz'][ti]; im=ax.imshow(mzp,cmap='RdBu_r',vmin=-1,vmax=1,origin='lower'); plt.colorbar(im,ax=ax,fraction=0.046)
    if 'full' in res:
        fp=res['full'][ti]; ax.quiver(XX,YY,fp[::s,::s,0],fp[::s,::s,1],color='k',alpha=0.5,scale=8)
    ax.set_title(f'RECON {method} acc={res["acc"]*100:.1f}%',color='#e6edf3',fontsize=10); ax.axis('off')
    fig.suptitle(f'Test run — {method}',color='#e6edf3',fontsize=12,fontweight='bold')
    fn=f'testrun_{method.replace(" ","_").replace("→","_to_")}.png'
    plt.savefig(fn,dpi=140,bbox_inches='tight',facecolor=fig.get_facecolor()); plt.show(); print("saved",fn)


## Section 11 — Master comparison + accuracy bars

In [ ]:
methods=list(results.keys())
fig=plt.figure(figsize=(16,3.0*len(methods))); fig.patch.set_facecolor('#0d1117')
gs=gridspec.GridSpec(len(methods),3,hspace=0.4,wspace=0.2)
for r,method in enumerate(methods):
    res=results[method]
    ax=fig.add_subplot(gs[r,0]); ax.set_facecolor('#161b22'); ax.imshow(mz,cmap='RdBu_r',vmin=-1,vmax=1,origin='lower')
    ax.set_ylabel(method,color='#e6edf3',fontsize=9,rotation=0,ha='right',va='center'); ax.set_xticks([]); ax.set_yticks([])
    if r==0: ax.set_title('TRUE Mz',color='#e6edf3',fontsize=10)
    ax=fig.add_subplot(gs[r,1]); ax.set_facecolor('#161b22'); ax.imshow(res['mz'][ti],cmap='RdBu_r',vmin=-1,vmax=1,origin='lower'); ax.axis('off')
    if r==0: ax.set_title('RECON Mz',color='#e6edf3',fontsize=10)
    ax=fig.add_subplot(gs[r,2]); ax.set_facecolor('#161b22')
    err=(np.sign(res['mz'][ti])!=np.sign(mz)).astype(float); ax.imshow(err,cmap='hot_r',vmin=0,vmax=1,origin='lower')
    ax.text(0.5,-0.12,f'acc {res["acc"]*100:.1f}%',transform=ax.transAxes,color='#e6edf3',ha='center',fontsize=9); ax.axis('off')
    if r==0: ax.set_title('ERROR',color='#e6edf3',fontsize=10)
fig.suptitle('All 7 pipelines — Mz reconstruction',color='#e6edf3',fontsize=14,fontweight='bold')
plt.savefig('fig_master_comparison.png',dpi=140,bbox_inches='tight',facecolor=fig.get_facecolor()); plt.show(); print("saved master")

fig,ax=plt.subplots(figsize=(11,5)); fig.patch.set_facecolor('#0d1117'); ax.set_facecolor('#161b22')
ax.tick_params(colors='#c9d1d9'); [s.set_color('#30363d') for s in ax.spines.values()]
names=list(results.keys()); accs=[results[k]['acc']*100 for k in names]
cols=['#58a6ff' if k=='CNN alone' else '#f78166' if 'alone' in k else '#3fb950' for k in names]
ax.bar(range(len(names)),accs,color=cols); ax.axhline(50,color='#8b949e',ls=':',label='random')
ax.set_xticks(range(len(names))); ax.set_xticklabels(names,rotation=30,ha='right',color='#c9d1d9')
ax.set_ylabel('Mz domain accuracy (%)',color='#c9d1d9'); ax.set_title('CNN vs classical vs hybrid',color='#e6edf3',fontweight='bold')
ax.legend(framealpha=0.3,labelcolor='#c9d1d9',facecolor='#161b22')
plt.tight_layout(); plt.savefig('fig_accuracy_bars.png',dpi=140,bbox_inches='tight',facecolor=fig.get_facecolor()); plt.show(); print("saved bars")


## Section 12 — Training curves

In [ ]:
fig,ax=plt.subplots(figsize=(10,5)); fig.patch.set_facecolor('#0d1117'); ax.set_facecolor('#161b22')
ax.tick_params(colors='#c9d1d9'); [s.set_color('#30363d') for s in ax.spines.values()]
ax.plot(hist_cnn['va'],lw=2,label='CNN alone',color='#58a6ff')
for name in helpers: ax.plot(helper_hist[name]['va'],lw=2,label=f'{name}→CNN')
ax.set_xlabel('epoch',color='#c9d1d9'); ax.set_ylabel('val MSE',color='#c9d1d9')
ax.set_title('Validation curves: CNN vs helper-augmented',color='#e6edf3',fontweight='bold')
ax.legend(framealpha=0.3,labelcolor='#c9d1d9',facecolor='#161b22')
plt.tight_layout(); plt.savefig('fig_training_curves.png',dpi=140,bbox_inches='tight',facecolor=fig.get_facecolor()); plt.show(); print("saved curves")


## Section 13 — Export for report

In [ ]:
with open('results_table.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['method','mz_acc_pct','vec_ang_deg','mse'])
    for k,v in results.items():
        w.writerow([k,f"{v['acc']*100:.2f}",f"{v['ang']:.2f}" if v['ang'] else "",f"{v['mse']:.5f}" if v['mse'] else ""])
np.savez('reconstructions.npz',true_mx=mx,true_my=my,true_mz=mz,
         **{k.replace(' ','_').replace('→','_to_'):v['mz'][ti] for k,v in results.items()})
np.savez('training_histories.npz',cnn_va=hist_cnn['va'],**{f'{n}_va':helper_hist[n]['va'] for n in helpers})
with open('run_config.json','w') as f:
    json.dump(dict(GRID_N=GRID_N,N_ANGLES=N_ANGLES,angles=ANGLE_LIST,C_CONST=C_CONST,
                   N_TRAIN=N_TRAIN,EPOCHS=EPOCHS,WALL_TYPE=WALL_TYPE),f,indent=2,default=str)
print("wrote results_table.csv, reconstructions.npz, training_histories.npz, run_config.json")
try:
    from google.colab import files
    for fn in ['results_table.csv','reconstructions.npz','training_histories.npz','run_config.json',
               'fig_master_comparison.png','fig_accuracy_bars.png','fig_training_curves.png','fig0_diagnostic.png']:
        files.download(fn)
except: print("(download in file browser)")
